In [1]:

import os
import platform
import subprocess

print("Python:", platform.python_version())
print("CPU cores:", os.cpu_count())

try:
    result = subprocess.run(
        ["nvidia-smi"],
        capture_output=True,
        text=True,
        timeout=10
    )
    print(result.stdout if result.returncode == 0
          else "No NVIDIA GPU detected in this runtime.")
except Exception as e:
    print("GPU check:", e)



Python: 3.13.15
CPU cores: 2
Mon Oct 19 11:33:13 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   36C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+------------------

In [ ]:

import os

print("Running in Google Colab:", "COLAB_RELEASE_TAG" in os.environ)
print("Hostname:", os.uname().nodename)
print("CPU cores:", os.cpu_count())


Running in Google Colab: True
Hostname: 64f887808ce5
CPU cores: 2


In [2]:

import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "VRAM (GiB):",
        round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2)
    )


PyTorch: 2.11.0+cu130
CUDA available: True
GPU: Tesla T4
VRAM (GiB): 14.56


In [3]:

import sys
import torch

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA build:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0))

try:
    import vllm
    print("vLLM already installed:", vllm.__version__)
except ImportError:
    print("vLLM is not installed yet.")


Python: 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]
PyTorch: 2.11.0+cu130
CUDA build: 13.0
GPU: Tesla T4
vLLM is not installed yet.


In [4]:

import sys
import torch
import subprocess

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("PyTorch CUDA build:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0))

result = subprocess.run(
    ["nvidia-smi", "--query-gpu=driver_version", "--format=csv,noheader"],
    capture_output=True,
    text=True
)
print("NVIDIA driver:", result.stdout.strip() or result.stderr.strip())

try:
    import vllm
    print("vLLM installed:", vllm.__version__)
except ImportError:
    print("vLLM installed: No")


Python: 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]
PyTorch: 2.11.0+cu130
PyTorch CUDA build: 13.0
GPU: Tesla T4
NVIDIA driver: 580.82.07
vLLM installed: No


In [5]:

!pip -q install uv


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.3/17.3 MB 97.1 MB/s eta 0:00:00


In [6]:

!uv venv --python 3.12 --seed /content/vllm-env


Using CPython 3.12.3 interpreter at: /usr/bin/python3.12
Creating virtual environment with seed packages at: vllm-env
 + pip==26.2.1
Activate with: source vllm-env/bin/activate


In [7]:

! /content/vllm-env/bin/python --version
! /content/vllm-env/bin/python -c "import sys; print(sys.executable)"


Python 3.12.3
/content/vllm-env/bin/python


In [8]:

!uv pip install --python /content/vllm-env/bin/python vllm --torch-backend=auto


Using Python 3.12.3 environment at: vllm-env
Resolved 198 packages in 1.97s
Prepared 198 packages in 1m 25s
Installed 198 packages in 1.98s
 + agent-detector==2.0.0
 + aiohappyeyeballs==2.7.1
 + aiohttp==3.14.5
 + aiosignal==1.4.0
 + annotated-doc==0.0.5
 + annotated-types==0.8.0
 + anthropic==1.15.0
 + anyio==4.15.1
 + apache-tvm-ffi==0.1.11
 + astor==0.8.1
 + attrs==26.1.0
 + blake3==1.0.11
 + cachetools==7.2.3
 + cbor2==6.1.5
 + certifi==2026.10.18
 + cffi==2.1.1
 + charset-normalizer==3.5.2
 + click==8.5.0
 + cloudpickle==3.1.2
 + compressed-tensors==0.17.0
 + cryptography==50.0.2
 + cuda-bindings==13.4.4
 + cuda-core==1.2.2
 + cuda-pathfinder==1.8.3
 + cuda-python==13.4.4
 + cuda-tile==1.6.0
 + cuda-toolkit==13.2.1
 + depyf==0.20.0
 + detect-installer==0.2.1
 + dill==0.4.1
 + dnspython==2.9.0
 + docstring-parser==0.18.0
 + einops==0.8.2
 + email-validator==2.3.0
 + fastapi==0.136.3
 + fastapi-cli==0.0.32
 + fastapi-cloud-cli==0.26.0
 + fastar==0.12.0
 + fastsafetensors==0.5.0
 + f

In [9]:

# Use the Python interpreter from our isolated vLLM environment.
# This avoids accidentally testing Colab's original Python 3.13 environment.

import subprocess

# Check that vLLM and its PyTorch dependency can be imported.
check_code = """
import torch
import vllm

print("vLLM version:", vllm.__version__)
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    # Initialize CUDA and query the GPU.
    torch.cuda.init()
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "Total GPU memory (GiB):",
        round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2)
    )
"""

# Run the checks using the isolated Python 3.12 environment.
result = subprocess.run(
    ["/content/vllm-env/bin/python", "-c", check_code],
    capture_output=True,
    text=True
)

print(result.stdout)

# Display any import or CUDA errors for diagnosis.
if result.returncode != 0:
    print("Verification failed:")
    print(result.stderr)


vLLM version: 0.32.0
PyTorch version: 2.13.0+cu132
CUDA available: True
GPU: Tesla T4
Total GPU memory (GiB): 14.56



In [10]:
# Why: Verify that the isolated vLLM installation exposes its CLI
# before we spend time downloading model weights.

import subprocess

result = subprocess.run(
    ["/content/vllm-env/bin/vllm", "--version"],
    capture_output=True,
    text=True,
    timeout=30
)

print("Return code:", result.returncode)
print("Output:", result.stdout.strip())

if result.returncode != 0:
    print("Error:", result.stderr.strip())

Return code: 0
Output: 0.32.0


In [11]:
# Why: Start an OpenAI-compatible inference API using vLLM.
# The server runs in the background so we can send requests from
# another notebook cell without blocking the notebook.

import subprocess
import time
import os

# Keep logs so we can diagnose startup or CUDA errors.
log_file = open("/content/vllm-server.log", "w")

command = [
    "/content/vllm-env/bin/vllm",
    "serve",
    "Qwen/Qwen2.5-1.5B-Instruct",
    "--host", "127.0.0.1",
    "--port", "8000",
    "--dtype", "half",
    "--max-model-len", "2048",
    "--gpu-memory-utilization", "0.80",
]

server = subprocess.Popen(
    command,
    stdout=log_file,
    stderr=subprocess.STDOUT,
    start_new_session=True,
)

# Give the process a little time to begin initialization.
time.sleep(10)

print("Server process ID:", server.pid)
print("Process running:", server.poll() is None)
print("Recent startup logs:")

with open("/content/vllm-server.log", "r") as f:
    print(f.read()[-6000:])

Server process ID: 14527
Process running: True
Recent startup logs:



In [12]:
# Why: Check the server's health endpoint without starting another server.
# This tells us whether vLLM has finished loading and is accepting requests.

import time
import requests
import subprocess

url = "http://127.0.0.1:8000/health"

for attempt in range(6):
    try:
        response = requests.get(url, timeout=5)
        print("HTTP status:", response.status_code)
        print("Response:", response.text[:1000])

        if response.status_code == 200:
            print("✅ Server is healthy.")
            break
    except requests.RequestException as e:
        print(f"Attempt {attempt + 1}: API not ready yet ({e})")

    time.sleep(10)

# Show recent logs if the server hasn't become ready.
print("\n--- Recent server logs ---")
with open("/content/vllm-server.log", "r") as f:
    print(f.read()[-8000:])

# Check whether the server process exited.
print("\nProcess status:", server.poll())

Attempt 1: API not ready yet (HTTPConnectionPool(host='127.0.0.1', port=8000): Max retries exceeded with url: /health (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x790ad03e1a90>: Failed to establish a new connection: [Errno 111] Connection refused')))
Attempt 2: API not ready yet (HTTPConnectionPool(host='127.0.0.1', port=8000): Max retries exceeded with url: /health (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x790ad107fb10>: Failed to establish a new connection: [Errno 111] Connection refused')))
Attempt 3: API not ready yet (HTTPConnectionPool(host='127.0.0.1', port=8000): Max retries exceeded with url: /health (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x790ad0218050>: Failed to establish a new connection: [Errno 111] Connection refused')))
Attempt 4: API not ready yet (HTTPConnectionPool(host='127.0.0.1', port=8000): Max retries exceeded with url: /health (Caused by NewConnectionError(

In [13]:
# Why: Determine whether vLLM is still initializing, ready, or stuck.
# We inspect the existing server rather than launching another one.

import subprocess
import requests
import time

# Check whether the API is responding.
try:
    response = requests.get(
        "http://127.0.0.1:8000/health",
        timeout=5
    )
    print("Health endpoint:", response.status_code, response.text[:500])
except requests.RequestException as exc:
    print("Health endpoint not ready:", exc)

# Inspect the most recent server messages.
print("\n--- Latest vLLM logs ---")
with open("/content/vllm-server.log", "r") as f:
    print(f.read()[-10000:])

# Inspect current GPU memory usage.
print("\n--- GPU status ---")
subprocess.run(["nvidia-smi"])

Health endpoint: 200 

--- Latest vLLM logs ---
ore pid=14684) INFO 10-19 12:21:02 [kernel_warmup.py:189] JIT kernel warmup finished in 0.00s.

Capturing CUDA graphs (PIECEWISE): 100%|██████████| 51/51 [00:03<00:00, 13.07it/s]

Capturing CUDA graphs (FULL): 100%|██████████| 35/35 [00:01<00:00, 19.01it/s]
(EngineCore pid=14684) INFO 10-19 12:21:33 [model_runner.py:1127] Graph capturing finished in 7 secs, took 0.15 GiB
(EngineCore pid=14684) INFO 10-19 12:21:33 [gpu_worker.py:971] CUDA graph pool memory: 0.15 GiB (actual), 0.34 GiB (estimated), difference: 0.19 GiB (130.3%).
(EngineCore pid=14684) INFO 10-19 12:21:33 [gpu_worker.py:1034] Free memory on device (14.46/14.56 GiB) on startup. Desired GPU memory utilization is (0.8, 11.65 GiB). Actual usage is 3.24 GiB for consumed memory (weights + non-torch), 0.78 GiB for peak activation, and 0.15 GiB for CUDAGraph memory. Replace gpu_memory_utilization config with `--kv-cache-memory=7870882919` (7.33 GiB) to fit into requested memory, or 

CompletedProcess(args=['nvidia-smi'], returncode=0)

In [14]:
# Why: Send a request to vLLM's OpenAI-compatible chat API.
# We measure end-to-end request latency so we have a baseline
# for later performance experiments.

import requests
import time

url = "http://127.0.0.1:8000/v1/chat/completions"

payload = {
    "model": "Qwen/Qwen2.5-1.5B-Instruct",
    "messages": [
        {
            "role": "user",
            "content": "Explain continuous batching in LLM serving in three simple sentences."
        }
    ],
    "temperature": 0.2,
    "max_tokens": 120
}

start = time.perf_counter()

response = requests.post(
    url,
    json=payload,
    timeout=120
)

elapsed = time.perf_counter() - start

print("HTTP status:", response.status_code)
response.raise_for_status()

result = response.json()

print("\nModel response:")
print(result["choices"][0]["message"]["content"])

print("\n--- Request metrics ---")
print(f"End-to-end latency: {elapsed:.3f} seconds")
print("Prompt tokens:", result["usage"]["prompt_tokens"])
print("Completion tokens:", result["usage"]["completion_tokens"])
print("Total tokens:", result["usage"]["total_tokens"])

HTTP status: 200

Model response:
Continuous Batching in Large Language Model (LLM) serving involves processing and responding to multiple requests simultaneously rather than sequentially, optimizing performance and reducing latency for users.

--- Request metrics ---
End-to-end latency: 1.492 seconds
Prompt tokens: 42
Completion tokens: 33
Total tokens: 75


In [15]:
# Why: Warm up the model to reduce first-request effects, then measure
# repeated requests to get a more useful latency baseline.

import requests
import time
import statistics

url = "http://127.0.0.1:8000/v1/chat/completions"

payload = {
    "model": "Qwen/Qwen2.5-1.5B-Instruct",
    "messages": [
        {
            "role": "user",
            "content": "Explain continuous batching in LLM serving."
        }
    ],
    "temperature": 0,
    "max_tokens": 100
}

# Warm-up request: do not include it in the measurements.
warmup = requests.post(url, json=payload, timeout=120)
warmup.raise_for_status()

latencies = []
completion_tokens = []

# Run five sequential requests for a simple baseline.
for i in range(5):
    start = time.perf_counter()

    response = requests.post(url, json=payload, timeout=120)
    response.raise_for_status()

    elapsed = time.perf_counter() - start
    data = response.json()

    latencies.append(elapsed)
    completion_tokens.append(data["usage"]["completion_tokens"])

    print(
        f"Request {i + 1}: "
        f"latency={elapsed:.3f}s, "
        f"completion_tokens={data['usage']['completion_tokens']}"
    )

print("\n--- Baseline summary ---")
print(f"Mean latency: {statistics.mean(latencies):.3f}s")
print(f"Median latency: {statistics.median(latencies):.3f}s")
pRequests: 5
$0rint(f"Mean completion tokens: {statistics.mean(completion_tokens):.1f}")
print("Requests:", len(latencies))

Request 1: latency=1.502s, completion_tokens=100
Request 2: latency=1.505s, completion_tokens=100
Request 3: latency=1.510s, completion_tokens=100
Request 4: latency=1.521s, completion_tokens=100
Request 5: latency=1.518s, completion_tokens=100

--- Baseline summary ---
Mean latency: 1.511s
Median latency: 1.510s
Mean completion tokens: 100.0
Requests: 5


In [16]:
# Why: Compare concurrent request handling with our sequential baseline.
# ThreadPoolExecutor submits four independent HTTP requests at once.
# vLLM decides how to schedule them internally.

import requests
import time
from concurrent.futures import ThreadPoolExecutor, as_completed

url = "http://127.0.0.1:8000/v1/chat/completions"

def send_request(request_id):
    # Give each request a distinct prompt.
    payload = {
        "model": "Qwen/Qwen2.5-1.5B-Instruct",
        "messages": [
            {
                "role": "user",
                "content": (
                    f"Request {request_id}: Explain one benefit of "
                    "continuous batching in two short sentences."
                )
            }
        ],
        "temperature": 0,
        "max_tokens": 100
    }

    start = time.perf_counter()
    response = requests.post(url, json=payload, timeout=120)
    elapsed = time.perf_counter() - start
    response.raise_for_status()

    data = response.json()

    return {
        "request_id": request_id,
        "latency_seconds": elapsed,
        "completion_tokens": data["usage"]["completion_tokens"],
        "answer": data["choices"][0]["message"]["content"]
    }

results = []
wall_start = time.perf_counter()

# Submit four requests concurrently.
with ThreadPoolExecutor(max_workers=4) as executor:
    futures = [
        executor.submit(send_request, i)
        for i in range(1, 5)
    ]

    for future in as_completed(futures):
        results.append(future.result())

wall_seconds = time.perf_counter() - wall_start
total_tokens = sum(r["completion_tokens"] for r in results)

print("--- Individual requests ---")
for r in sorted(results, key=lambda x: x["request_id"]):
    print(
        f"Request {r['request_id']}: "
        f"latency={r['latency_seconds']:.3f}s, "
        f"tokens={r['completion_tokens']}"
    )

print("\n--- Concurrent benchmark ---")
print(f"Concurrent requests: {len(results)}")
print(f"Group wall-clock time: {wall_seconds:.3f}s")
print(f"Total output tokens: {total_tokens}")
print(f"Aggregate output throughput: {total_tokens / wall_seconds:.2f} tokens/s")

--- Individual requests ---
Request 1: latency=0.662s, tokens=39
Request 2: latency=0.796s, tokens=47
Request 3: latency=0.796s, tokens=47
Request 4: latency=0.795s, tokens=47

--- Concurrent benchmark ---
Concurrent requests: 4
Group wall-clock time: 0.802s
Total output tokens: 180
Aggregate output throughput: 224.57 tokens/s


In [17]:
# Why: Streaming lets us observe when the first generated text arrives
# and how long the complete streamed response takes.
# This is an application-level approximation of TTFT, not an exact
# measurement of when the model generated its first token internally.

import requests
import time
import json
import statistics

url = "http://127.0.0.1:8000/v1/chat/completions"

payload = {
    "model": "Qwen/Qwen2.5-1.5B-Instruct",
    "messages": [
        {
            "role": "user",
            "content": "Explain how Kubernetes schedules GPU workloads in five concise bullet points."
        }
    ],
    "temperature": 0,
    "max_tokens": 120,
    "stream": True,
    "stream_options": {"include_usage": True}
}

start = time.perf_counter()
first_content_time = None
last_content_time = None
content_chunks = 0
content_parts = []
usage = None

with requests.post(
    url,
    json=payload,
    stream=True,
    timeout=120
) as response:
    print("HTTP status:", response.status_code)
    response.raise_for_status()

    for line in response.iter_lines(decode_unicode=True):
        if not line or not line.startswith("data: "):
            continue

        data = line[6:]
        if data == "[DONE]":
            break

        event = json.loads(data)

        # The final streaming event may contain usage information.
        if event.get("usage"):
            usage = event["usage"]

        choices = event.get("choices", [])
        if not choices:
            continue

        delta = choices[0].get("delta", {})
        content = delta.get("content")

        if content:
            now = time.perf_counter()

            if first_content_time is None:
                first_content_time = now

            last_content_time = now
            content_chunks += 1
            content_parts.append(content)

end = time.perf_counter()

print("\n--- Streaming result ---")
print("Response text:")
print("".join(content_parts))

if first_content_time is not None:
    print(f"\nTime to first content: {first_content_time - start:.3f}s")
    print(f"Total streaming duration: {end - start:.3f}s")
    print("Content chunks received:", content_chunks)

    if content_chunks > 1:
        print(
            "Average interval between content chunks:",
            f"{(last_content_time - first_content_time) / (content_chunks - 1):.4f}s"
        )

print("Token usage:", usage)

HTTP status: 200

--- Streaming result ---
Response text:
- Kubernetes uses the `nodeSelector` and `tolerations` features to ensure that only nodes with available GPUs meet the workload's requirements.
- Nodes marked as "GPU-ready" or "gpu-enabled" can host GPU-based applications.
- The scheduler prioritizes pods on nodes with matching node labels for GPU availability.
- Pods with GPU requests (e.g., `resources.requests[cpu] > 0 && resources.requests[nvidia.com/gpu] > 0`) are scheduled first based on resource needs.
- Once a suitable node is found, the scheduler assigns the pod to it, ensuring

Time to first content: 0.124s
Total streaming duration: 1.952s
Content chunks received: 120
Average interval between content chunks: 0.0154s
Token usage: {'prompt_tokens': 43, 'total_tokens': 163, 'completion_tokens': 120}


In [18]:
# Why: Compare concurrency levels using the same workload.
# We warm up the server first and then run separate batches at
# concurrency 1, 2, and 4 to observe scaling behavior.

import requests
import time
import statistics
from concurrent.futures import ThreadPoolExecutor, as_completed

URL = "http://127.0.0.1:8000/v1/chat/completions"
MODEL = "Qwen/Qwen2.5-1.5B-Instruct"

def make_request(request_id):
    payload = {
        "model": MODEL,
        "messages": [
            {
                "role": "user",
                "content": (
                    "Explain why continuous batching improves LLM serving "
                    "efficiency. Give exactly three concise bullet points."
                )
            }
        ],
        "temperature": 0,
        "max_tokens": 80
    }

    start = time.perf_counter()
    response = requests.post(URL, json=payload, timeout=120)
    latency = time.perf_counter() - start
    response.raise_for_status()

    data = response.json()

    return {
        "request_id": request_id,
        "latency": latency,
        "tokens": data["usage"]["completion_tokens"]
    }

# Warm-up request to reduce first-request effects.
make_request("warmup")

for concurrency in (1, 2, 4):
    results = []
    wall_start = time.perf_counter()

    with ThreadPoolExecutor(max_workers=concurrency) as executor:
        futures = [
            executor.submit(make_request, i)
            for i in range(concurrency)
        ]

        for future in as_completed(futures):
            results.append(future.result())

    wall_time = time.perf_counter() - wall_start
    total_tokens = sum(r["tokens"] for r in results)

    latencies = [r["latency"] for r in results]

    print(f"\n--- Concurrency: {concurrency} ---")
    print(f"Batch wall time: {wall_time:.3f}s")
    print(f"Mean request latency: {statistics.mean(latencies):.3f}s")
    print(f"Total output tokens: {total_tokens}")
    print(f"Aggregate throughput: {total_tokens / wall_time:.2f} tokens/s")


--- Concurrency: 1 ---
Batch wall time: 0.794s
Mean request latency: 0.791s
Total output tokens: 50
Aggregate throughput: 63.01 tokens/s

--- Concurrency: 2 ---
Batch wall time: 0.776s
Mean request latency: 0.774s
Total output tokens: 100
Aggregate throughput: 128.89 tokens/s

--- Concurrency: 4 ---
Batch wall time: 0.807s
Mean request latency: 0.800s
Total output tokens: 200
Aggregate throughput: 247.87 tokens/s


In [19]:
# Why: Colab provides a temporary cloud runtime, not a Kubernetes
# cluster by default. Check which tools are already available before
# choosing a suitable learning environment.

import shutil
import subprocess
import os

# Check whether the required command-line tools exist.
for tool in ("kubectl", "docker", "kind", "minikube", "helm"):
    print(f"{tool}: {shutil.which(tool) or 'not installed'}")

# Check whether a Kubernetes configuration is present.
kubeconfig = os.environ.get("KUBECONFIG")
default_config = os.path.expanduser("~/.kube/config")

print("\nKUBECONFIG environment variable:", kubeconfig or "not set")
print("Default kubeconfig exists:", os.path.exists(default_config))

# Check whether kubectl can reach a configured cluster.
if shutil.which("kubectl"):
    result = subprocess.run(
        ["kubectl", "cluster-info"],
        capture_output=True,
        text=True,
        timeout=15
    )
    print("\nkubectl cluster-info exit code:", result.returncode)
    print((result.stdout + result.stderr)[-2000:])

kubectl: not installed
docker: not installed
kind: not installed
minikube: not installed
helm: not installed

KUBECONFIG environment variable: not set
Default kubeconfig exists: False


In [5]:
!git config --global user.name "Subham999100"
!git config --global user.email "patnaiksubham31@gmail.com"

!git clone https://github.com/Subham999100/llm-inference-vllm-kubernetes-lab.git

Cloning into 'llm-inference-vllm-kubernetes-lab'...


In [6]:
!ls -la /content

total 24
drwxr-xr-x 1 root root 4096 Oct 19 13:42 .
drwxr-xr-x 1 root root 4096 Oct 19 13:16 ..
drwxr-xr-x 4 root root 4096 Sep 30 13:28 .config
drwx------ 6 root root 4096 Oct 19 13:40 drive
drwxr-xr-x 3 root root 4096 Oct 19 13:42 llm-inference-vllm-kubernetes-lab
drwxr-xr-x 1 root root 4096 Sep 30 13:28 sample_data


In [8]:
from google.colab import files
files.download('/content/llm-inference-vllm-kubernetes-lab/README.md')

FileNotFoundError: Cannot find file: /content/llm-inference-vllm-kubernetes-lab/README.md

In [ ]:
from google.colab import _message

result = _message.blocking_request("get_ipynb", timeout_sec=30)
print(str(result)[:1000])